# 2026-09-01_1 分组定义加固（方案 A）：DAL 增加 effect-size 判据
规则（a priori，三组统一施加）：Old 失显著 且 retention >= 0.5 且 |Old log2FC| >= 1 -> 移入中间类
retention = Old signed 效应量 / Young signed 效应量（signed：Macular 用 +log2FC，Peripheral 用 -log2FC）
DM / AAM 仅验证（预期零受影响），不改动基因名单。原分类主表只读不动。


In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(readr)
})

date_str <- "2026-09-01_1"
data_dir <- "/project/work/2026-09-01_DEG_summary/Data/"

trajectory_file <- "/project/work/2026-07-29_trajectory_class/Data/2026-07-29_1_five_stage_regional_DEG_trajectory_classified.csv"

retention_cutoff <- 0.5
old_log2fc_cutoff <- 1


In [ ]:
# ---- 读入主表，计算 signed 效应量与 retention --------------------------------
trajectory_df <- read_csv(trajectory_file, show_col_types = FALSE) %>%
  mutate(
    direction_sign = case_when(
      direction_class == "Macular direction" ~ 1,
      direction_class == "Peripheral direction" ~ -1,
      TRUE ~ NA_real_
    ),
    Young_signed_effect = Young_log2FoldChange * direction_sign,
    Old_signed_effect = Old_log2FoldChange * direction_sign,
    effect_retention = Old_signed_effect / Young_signed_effect
  )


In [ ]:
# ---- DAL 应用判据：维持效应但失显著 -> 中间类 --------------------------------
trajectory_refined <- trajectory_df %>%
  mutate(
    maintained_effect = !is.na(effect_retention) &
      effect_retention >= retention_cutoff &
      abs(Old_signed_effect) >= old_log2fc_cutoff,
    subclass_refined = if_else(
      subclass == "Developmental aging-lost" & maintained_effect,
      "Maintained-effect sub-threshold",
      subclass
    ),
    refinement_flag = if_else(
      subclass == "Developmental aging-lost" & maintained_effect,
      "moved_from_DAL",
      "unchanged"
    )
  )


In [ ]:
# ---- 核查 -------------------------------------------------------------------
# 1) DM / AAM 在同一判据下零受影响（DM 全部天然满足维持；AAM 效应量由 M 判定保证）
dm_check <- trajectory_refined %>%
  filter(subclass == "Developmental maintained") %>%
  summarise(
    n = n(),
    n_fail_retention = sum(effect_retention < retention_cutoff | abs(Old_signed_effect) < old_log2fc_cutoff)
  )
aam_check <- trajectory_refined %>%
  filter(subclass == "Adult acquired and maintained") %>%
  summarise(n = n(), n_old_effect_below_M_cutoff = sum(abs(Old_signed_effect) <= 0.5))
cat("DM 基因数:", dm_check$n, "; 未通过判据:", dm_check$n_fail_retention, "\n")
cat("AAM 基因数:", aam_check$n, "; Old 效应量低于 M 阈值:", aam_check$n_old_effect_below_M_cutoff, "\n")
stopifnot(dm_check$n_fail_retention == 0, aam_check$n_old_effect_below_M_cutoff == 0)

# 2) DAL 代表 loci 不受影响
loci_check <- trajectory_refined %>%
  filter(gene %in% c("CYP26A1", "APOE", "MAN1C1"), subclass == "Developmental aging-lost")
stopifnot(all(loci_check$subclass_refined == "Developmental aging-lost"))
cat("DAL 代表 loci（CYP26A1/APOE/MAN1C1）全部保留在 DAL。\n")


In [ ]:
# ---- 输出 -------------------------------------------------------------------
moved_df <- trajectory_refined %>%
  filter(refinement_flag == "moved_from_DAL") %>%
  select(
    celltype, gene, direction_class, subclass, subclass_refined,
    Young_signed_effect, Old_signed_effect, effect_retention,
    Young_padj, Old_padj, Early_Mid_Late_Young_Old
  ) %>%
  arrange(celltype, desc(effect_retention))

refined_counts <- trajectory_refined %>%
  filter(subclass_refined %in% c(
    "Developmental maintained",
    "Developmental aging-lost",
    "Adult acquired and maintained",
    "Maintained-effect sub-threshold"
  )) %>%
  count(subclass_refined, direction_class, celltype, name = "n_genes") %>%
  arrange(subclass_refined, direction_class, celltype)

write_csv(trajectory_refined, file.path(data_dir, paste0(date_str, "_five_stage_trajectory_refined.csv")), na = "NA")
write_csv(moved_df, file.path(data_dir, paste0(date_str, "_DAL_subthreshold_gene_list.csv")), na = "NA")
write_csv(refined_counts, file.path(data_dir, paste0(date_str, "_subclass_refined_counts.csv")), na = "NA")


In [ ]:
# ---- 汇总与参数记录 ---------------------------------------------------------
n_dal_before <- sum(trajectory_refined$subclass == "Developmental aging-lost")
n_moved <- sum(trajectory_refined$refinement_flag == "moved_from_DAL")
cat("DAL 修正前:", n_dal_before, "; 移入中间类:", n_moved, "; DAL 剩余:", n_dal_before - n_moved, "\n")
print(count(filter(trajectory_refined, refinement_flag == "moved_from_DAL"), direction_class, celltype))

writeLines(
  c(
    paste0("date_str: ", date_str),
    paste0("trajectory_file: ", trajectory_file),
    paste0("retention_cutoff: ", retention_cutoff),
    paste0("old_log2fc_cutoff: ", old_log2fc_cutoff),
    "rule: DAL & effect_retention >= 0.5 & |Old_signed_effect| >= 1 -> Maintained-effect sub-threshold",
    "DM/AAM: verified unaffected under the same criterion; gene lists unchanged",
    "",
    capture.output(sessionInfo())
  ),
  file.path(data_dir, paste0(date_str, "_parameters_sessionInfo.txt"))
)
cat("完成。输出目录:", data_dir, "\n")
